# 03 – Flächenströme zwischen zwei gespeicherten Zuständen (mit Bestandsbilanz Neubau/Abriss)

Zustände A und B werden aus `results/szenarien/` gelesen (mit Integritätsprüfung) und **nie verändert**; das wird nach jedem Lauf über Dateihashes nachgewiesen.

Zwei Modi:

- `MODUS='VorhandeneFluesse'` (Standard): liest nur – registrierte Flussdatei prüfen oder gespeichertes Ergebnis laden. **Kein Solver.**
- `MODUS='Rekonstruktion'`: lineare Optimierung (konstanter Bestand je unveränderlichem Block; mit Neubau/Abriss ein gemeinsames LP über alle Blöcke), **nur mit `REKONSTRUIEREN=True`**.

Status: `NICHT_AUSGEFUEHRT`, `EINGABEN_UNGUELTIG`, `MACHBAR`, `NICHT_MACHBAR`, `NICHT_ABSCHLIESSEND_GELOEST` (Zeitlimit – kein Unmachbarkeitsnachweis), `BERECHNUNGSFEHLER`.
Bestandsbilanz: \(T_B = T_A - D + N\). Neubau \(N\) und Abriss \(D\) werden nur aus ausdrücklichen Bruttovorgaben in `data/eingaben/bestandsentwicklung/Bestandsentwicklung.xlsx` übernommen (Rate %/a, Periodenfläche oder ausdrücklich gewählter Bilanzrest). Ist nur die Nettoänderung bekannt, wird nicht rekonstruiert (keine Annahme \(N = D = 0\)). Neubau und Abriss sind externe Flüsse, keine Gebäudetypen. Die Rechenlogik liegt in `modell/flaechenstroeme.py`, `modell/bestandsbilanz.py` und `modell/massnahmen.py` (Maßnahmenraten).

Ablauf: Paar und Modus wählen (Steuerung) → 1–2 Zustände, Zeitspanne und Bilanzgrundlage → 3 Regeln und Blockdiagnose → 4–5 Flüsse laden oder Rekonstruktion ausdrücklich starten, Status und Kontrollen → 6 Matrizen und Sankey → 7 Änderungsanteile, Neubau und Abriss → 8 Maßnahmenraten → 9 Wiederholbarkeit gegen ein gespeichertes Ergebnis (nur nach neuer Rekonstruktion) → 10 optional speichern und wieder laden.

## Start

- **Was:** Projektordner finden und Module laden.
- **Eingaben:** `config/projekt.json`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** –
- **Sperrende Fehler:** Projektordner nicht gefunden.

In [1]:
# Einheitliche Startzelle: Projektordner suchen, aktuelle Modellmodule laden, Fassung prüfen, Projektpfade lesen.
import os, sys
from pathlib import Path
try:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config' / 'projekt.json').is_file())
except StopIteration:
    raise FileNotFoundError('Projektordner nicht gefunden: Das Notebook muss im Ordner notebooks/ des Projekts liegen (config/projekt.json fehlt).') from None
MODELL = ROOT / 'modell'
if str(MODELL) not in sys.path:
    sys.path.insert(0, str(MODELL))
for _name in [n for n, m in list(sys.modules.items()) if os.path.normcase(str(getattr(m, '__file__', None) or '')).startswith(os.path.normcase(str(MODELL)))]:
    del sys.modules[_name]   # nach einer Aktualisierung immer die Module aus dem Projektordner laden (kein veralteter Kernelstand)
import fassung
print('Projektordner:', ROOT)
print(fassung.pruefe_notebook('03_Flaechenstroeme', '1.6.3-AP6'))
import gebaeudesektor as gs
from IPython.display import display
PROJEKT = gs.lade_projekt(ROOT)
import flaechenstroeme as fs
import bestandsbilanz as bb
import massnahmen as ms
import abnahme as ab

Projektordner: c:\Users\ma0026.NB-MA0026\Documents\peq-higrube\Berechnungen\gebaeudesektormodell\Gebaeudesektor_konsolidiert\Gebaeudesektor
Fassung 1.6.3-AP6 (Notebook und Modell passen zusammen)


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Registrierte Rekonstruktionen

- **Was:** Paare A/B mit Regeldatei und optionaler Flussdatei; gespeicherte Szenarien.
- **Eingaben:** `config/flaechenstroeme/rekonstruktionen.json`, `results/szenarien/`
- **Rechnet oder liest:** liest nur
- **Annahmen:** –
- **Interpretation:** –
- **Sperrende Fehler:** ungültiges Register

In [2]:
display(fs.registrierte_rekonstruktionen(PROJEKT))
display(gs.ergebnisuebersicht(PROJEKT))

,Rekonstruktion_ID,A,B,Regeln,Flussdatei,Beschreibung
0,Bestand_2024__Transition_2040,Bestand_2024,Transition_2040,config/flaechenstroeme/uebergaenge.json,,Konstanter Bestand 2024 → Transition 2040
1,Bestand_2024__Transition_2040_Wachstum,Bestand_2024,Transition_2040_Wachstum,config/flaechenstroeme/uebergaenge.json,,Bestand 2024 → Transition 2040 mit Flächenwach...
2,Bestand_2024__Transition_2040_ambitioniert,Bestand_2024,Transition_2040_ambitioniert,config/flaechenstroeme/uebergaenge.json,,Bestand 2024 → Transition 2040 mit Flächenwach...


,Szenario_ID,Ordner,Status,Jahr,NGF_m2,Positive_Typen,Zustand_ID,Energie,Kennwerte,Integritaet
0,Bestand_2024,results/szenarien/Bestand_2024,LESBAR,2024,889872558.0,14640,6d9c1ee883204a04,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
1,Transition_2040,results/szenarien/Transition_2040,LESBAR,2040,889872558.0,16320,e07bff047ea3de97,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
2,Transition_2040_ambitioniert,results/szenarien/Transition_2040_ambitioniert,LESBAR,2040,950000000.0,8160,7efe32919bdba57d,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)
3,Transition_2040_Wachstum,results/szenarien/Transition_2040_Wachstum,LESBAR,2040,950000000.0,15336,cbf1cf4d08e5df11,Ausgeführt,True,OK (SHA-256 der Ausgaben geprüft)


## Bestandsbilanz Neubau/Abriss

- **Was:** Bruttovorgaben je registriertem Paar, Umrechnung von Raten in Periodenflächen und Bilanzprüfung gegen die gespeicherten Gesamtflächen.
- **Eingaben:** `data/eingaben/bestandsentwicklung/Bestandsentwicklung.xlsx`, `results/szenarien/`
- **Rechnet oder liest:** liest und rechnet Bilanzen, keine Optimierung
- **Annahmen:** Rate %/a bezogen auf NGF A und die Periodenlänge (linear). Bilanzrest nur, wenn ausdrücklich so eingetragen. Freigaben legen fest, welche Typen abgerissen bzw. neu gebaut werden dürfen; Bezugsbasis: Abriss bezogen auf A, Neubau auf B.
- **Interpretation:** `BILANZ_OK`: Rekonstruktion mit Neubau/Abriss. `NUR_NETTO`: nur N − D bekannt – bei ungleicher NGF keine Rekonstruktion.
- **Sperrende Fehler:** `WIDERSPRUCH` (Vorgaben passen nicht zu A/B), `UNVOLLSTAENDIG`, ungültige Datei

In [3]:
display(bb.szenariobilanz(PROJEKT))
display(bb.freigabetabelle(bb.lade_bestandsentwicklung(PROJEKT)))

,Verbindung_ID,A,B,Jahre,NGF_A_m2,NGF_B_m2,Netto_m2,Neubau_m2,Abriss_m2,Neubau_pct_pro_Jahr,Abriss_pct_pro_Jahr,Status,Meldung
0,Bestand_2024__Transition_2040,Bestand_2024,Transition_2040,16,889872558.0,889872558.0,-1.192093e-07,NaN,NaN,NaN,NaN,NUR_NETTO,Keine Bruttoangaben: nur N − D = T_B − T_A bek...
1,Bestand_2024__Transition_2040_Wachstum,Bestand_2024,Transition_2040_Wachstum,16,889872558.0,950000000.0,6.012744e+07,80000000.0,19872558.0,0.561878,0.139574,BILANZ_OK,Neubau und Abriss vorgegeben; T_A − D + N = T_...
2,Bestand_2024__Transition_2040_ambitioniert,Bestand_2024,Transition_2040_ambitioniert,16,889872558.0,950000000.0,6.012744e+07,80000000.0,19872558.0,0.561878,0.139574,BILANZ_OK,Neubau und Abriss vorgegeben; T_A − D + N = T_...


,Art,Merkmal,Freigegeben,Alle
0,Neubau,geometry,"EFH_1G, GWB_7G, EFH_2G, NiWo_2G, NiWo_3G, RH_2...",True
1,Neubau,quality,"Neu_20cm, Neu_30cm",False
2,Neubau,energy,"WP_Sole, WP_Wasser, WP_Luft, WP_Sole_fc, DH, B...",False
3,Neubau,pv,"PV0, PV30, PV70, PV90",True
4,Neubau,flex,"noFlex, Flex",True
5,Neubau,weather,"Wien, Innsbruck",True
6,Neubau,mass,"Leicht, mittel, schwer",True
7,Abriss,geometry,"EFH_1G, GWB_7G, EFH_2G, NiWo_2G, NiWo_3G, RH_2...",True
8,Abriss,quality,"vor1919, 1919-44, 1945-59, 1960-79, 1980-89, 1...",False
9,Abriss,energy,"WP_Sole, WP_Wasser, WP_Luft, WP_Sole_fc, DH, B...",True


## Steuerung

- **Was:** Auswahl des Paares, Modus und Schalter.
- **Eingaben:** Rekonstruktions-ID
- **Rechnet oder liest:** –
- **Annahmen:** Schalter standardmäßig `False`; Standardmodus liest nur.
- **Interpretation:** –
- **Sperrende Fehler:** –

In [4]:
REKONSTRUKTION = "Bestand_2024__Transition_2040_ambitioniert"
MODUS = 'Rekonstruktion'   # 'VorhandeneFluesse' (nur lesen) oder 'Rekonstruktion' (lineare Optimierung)
REKONSTRUIEREN = True        # Rekonstruktion ausdrücklich starten (nur im MODUS 'Rekonstruktion')
ZEITLIMIT_S = None            # optionales Zeitlimit je Block in Sekunden; Überschreitung = NICHT_ABSCHLIESSEND_GELOEST
EXPORTIEREN = True           # Ergebnis nach results/flaechenstroeme/<REKONSTRUKTION>/ speichern
ERSETZEN = True              # vorhandenes Ergebnis ersetzen (nur bewusst setzen)

## 1–2 Zustände, Zeitspanne und Bilanzgrundlage

- **Was:** Lädt A und B mit Integritätsprüfung, die Bestandsbilanz des Paares und zeigt die Pflichtprüfungen vor jeder Rekonstruktion.
- **Eingaben:** `results/szenarien/<A>`, `<B>`, Bestandsentwicklung.xlsx
- **Rechnet oder liest:** liest nur
- **Annahmen:** Gesamtbilanz T_A − D + N = T_B. Zeitreihenfolge: Jahr B > Jahr A.
- **Interpretation:** Bei ungleicher NGF ohne Bruttoangaben: FEHLER (nur Nettoänderung bekannt).
- **Sperrende Fehler:** Zustand fehlt, Integrität verletzt, ungültige Typen, Bilanzwiderspruch

In [5]:
regeln = fs.lade_rekonstruktion(REKONSTRUKTION, PROJEKT)
a, b = fs.lade_zustaende_geprueft(regeln, PROJEKT)
print('A:', a['name'], a['jahr'], a['datei'], '| B:', b['name'], b['jahr'], b['datei'], '| Zeitspanne:', b['jahr'] - a['jahr'], 'Jahre')
bilanz = bb.bilanz_fuer(REKONSTRUKTION, PROJEKT, a['df'].NGF_m2.sum(), b['df'].NGF_m2.sum(), a['jahr'], b['jahr'])
print('Bilanzstatus:', bilanz['status'], '–', bilanz['meldung'])
display(bilanz['tabelle'])
display(fs.pruefe_eingaben(a, b, regeln, bilanz))

A: Bestand_2024 2024 results/szenarien/Bestand_2024/Verteilung.csv | B: Transition_2040_ambitioniert 2040 results/szenarien/Transition_2040_ambitioniert/Verteilung.csv | Zeitspanne: 16 Jahre
Bilanzstatus: BILANZ_OK – Neubau und Abriss vorgegeben; T_A − D + N = T_B innerhalb der Bilanztoleranz.


,Groesse,NGF_m2,Anteil_Ausgangs_NGF_pct,pct_Ausgangs_NGF_pro_Jahr,Eingabeart,Vorgabe
0,Ausgangsbestand T_A,889872558.0,100.000000,NaN,,NaN
1,Abriss D,19872558.0,2.233191,0.139574,Bilanzrest,NaN
2,Verbleibender Bestand T_A − D,870000000.0,97.766809,NaN,,NaN
3,Neubau N,80000000.0,8.990051,0.561878,Periodenflaeche_m2,80000000.0
4,Zielbestand Soll T_A − D + N,950000000.0,106.756860,NaN,,NaN
5,Zielbestand Ist T_B,950000000.0,106.756860,NaN,,NaN
6,Nettoänderung N − D = T_B − T_A,60127442.0,6.756860,0.422304,,NaN


,Pruefung,Status,Detail
0,Regeln gültig und dokumentiert,OK,config/flaechenstroeme/uebergaenge.json
1,"Zustand A gültig (Typen, Nichtnegativität, FW/...",OK,Bestand_2024
2,Zustand A gespeichert und unverändert (Ausgabe...,OK,OK (SHA-256 der Ausgaben geprüft)
3,"Zustand B gültig (Typen, Nichtnegativität, FW/...",OK,Transition_2040_ambitioniert
4,Zustand B gespeichert und unverändert (Ausgabe...,OK,OK (SHA-256 der Ausgaben geprüft)
5,Zeitreihenfolge (Jahr B > Jahr A),OK,2024 → 2040
6,Gesamtbilanz T_A − D + N = T_B,OK,"A 889,872,558.00 m² − Abriss 19,872,558.00 m² ..."


## 3 Regeln

- **Was:** Zulässige Einzelübergänge je Eigenschaft (ohne unveränderte Paare).
- **Eingaben:** `config/flaechenstroeme/uebergaenge.json`
- **Rechnet oder liest:** liest und validiert
- **Annahmen:** Unverändert ist erlaubt; nicht genannte Änderungen sind verboten; keine Verkettung von Gruppenregeln; WP nur in zulässigen Zielqualitäten.
- **Interpretation:** –
- **Sperrende Fehler:** ungültige Regeln

In [6]:
print('Unveränderlich:', ', '.join(regeln['unveraenderlich']), '| Zielfunktion:', fs.ZIELFUNKTION)
t = fs.regeltabelle(regeln)
display(t[~t.Unveraendert])

Unveränderlich: geometry, weather, mass | Zielfunktion: 2: max. unveränderte Fläche, dann min. Versorgungswechsel


,Eigenschaft,Pre,Post,Unveraendert
11,quality,1919-44,Neu_10cm,False
12,quality,1919-44,Neu_20cm,False
13,quality,1919-44,Neu_30cm,False
15,quality,1945-59,Neu_10cm,False
16,quality,1945-59,Neu_20cm,False
17,quality,1945-59,Neu_30cm,False
19,quality,1960-79,Neu_20cm,False
20,quality,1960-79,Neu_30cm,False
22,quality,1980-89,Neu_20cm,False
23,quality,1980-89,Neu_30cm,False


## Blockdiagnose

- **Was:** Notwendige Bedingungen je unveränderlichem Block (Geometrie × Wetter × Masse), dieselben wie vor der Rekonstruktion.
- **Eingaben:** Zustände A und B, Regeln, Bestandsbilanz des Paares
- **Rechnet oder liest:** rechnet einfache Bilanzen, keine Optimierung
- **Annahmen:** Mit Bruttovorgaben (`BILANZ_OK`) werden Neubau und Abriss innerhalb der Freigaben berücksichtigt; sonst gelten die Bedingungen für konstanten Bestand. Geprüft werden Einzelmerkmale und (ab 1.6.2) Kombinationen veränderlicher Merkmale, z. B. Qualität × Versorgung: Fläche einer Kombination kann nur aus derselben Kombination oder freigegebenem Neubau entstehen bzw. nur in sie übergehen oder freigegeben abgerissen werden. Gemeldet werden nur minimale Kombinationen. Notwendige, nicht hinreichende Bedingung; scheitert die Rekonstruktion trotzdem (Solverstatus 2), ortet eine Fehlmengenanalyse in der Diagnose die betroffenen Typen (Ebene „Fehlmengen A/B“).
- **Interpretation:** 0 Konflikte: notwendige Bedingungen erfüllt. Defizite zeigen Flächen, die mit Regeln und Freigaben nicht erreichbar sind; Eigenschaft „Kombination“ kennzeichnet Konflikte, die erst im Zusammenspiel mehrerer Merkmale entstehen. Defizite zeigen Flächen, die mit Regeln und Freigaben nicht erreichbar sind. Bei ungleicher NGF ohne Bruttovorgaben gibt es nur eine Meldung „nur Nettoänderung bekannt“ (keine Blockprüfung). Die Zeile „Grundlage“ nennt, ob Neubau/Abriss berücksichtigt wurden.
- **Sperrende Fehler:** Bestandsbilanz nicht `BILANZ_OK` bei ungleicher NGF

In [7]:
diagnose = fs.blockdiagnose(a, b, regeln, bilanz)
print(len(diagnose), 'Konflikte in', diagnose.attrs['bloecke'], 'Blöcken | Grundlage:', diagnose.attrs['grundlage'])
display(diagnose.head(20))

0 Konflikte in 0 Blöcken | Grundlage: mit Neubau/Abriss (Bestandsentwicklung, Status BILANZ_OK)


,Block,Ebene,Eigenschaft,Kategorie,Meldung,Defizit_m2


## 4–5 Flüsse laden oder rekonstruieren; Status und Kontrollen

- **Was:** Liest vorhandene Flüsse bzw. startet die Rekonstruktion ausdrücklich; zeigt Status, Pflichtprüfungen und Diagnose.
- **Eingaben:** wie oben; registrierte Flussdatei oder gespeichertes Ergebnis
- **Rechnet oder liest:** Solver nur bei `MODUS='Rekonstruktion'` und `REKONSTRUIEREN=True`
- **Annahmen:** Zielfunktion in zwei Stufen: (1) maximale vollständig unveränderte Typenfläche; (2) bei gleicher unveränderter Fläche minimale Fläche mit Versorgungswechsel (Entscheidung 17.11.2026). Für Sanierung, PV und Flex sowie die Auswahl innerhalb der Freigaben können weiterhin gleichwertige Lösungen existieren.
- **Interpretation:** Nur `MACHBAR` liefert Flüsse. `NICHT_ABSCHLIESSEND_GELOEST` ist kein Unmachbarkeitsnachweis.
- **Sperrende Fehler:** –

In [8]:
ergebnis = fs.analysiere_rekonstruktion(REKONSTRUKTION, PROJEKT, modus=MODUS, rekonstruieren=REKONSTRUIEREN, zeitlimit_s=ZEITLIMIT_S)
display(fs.zusammenfassung(ergebnis))
display(ergebnis['pruefungen'])
if ergebnis['status'] != fs.MACHBAR:
    display(ergebnis['diagnose'].head(30))
elif len(ergebnis['kontrollen']):
    display(ergebnis['kontrollen'].head(30))

,Rekonstruktion,Status,Bedeutung,Quelle,A,B,Jahre,Fluesse,NGF_m2,Unveraendert_m2,Abriss_m2,Neubau_m2,Bilanzstatus,Zustaende_unveraendert,Meldung
0,Bestand_2024__Transition_2040_ambitioniert,MACHBAR,Gültige Flüsse; alle Pflichtprüfungen bestanden.,rekonstruiert,Bestand_2024,Transition_2040_ambitioniert,16,21800,8.700000e+08,3.484013e+08,1.987256e+07,8.000000e+07,BILANZ_OK,True,


,Pruefung,Status,Detail
0,Regeln gültig und dokumentiert,OK,config/flaechenstroeme/uebergaenge.json
1,"Zustand A gültig (Typen, Nichtnegativität, FW/...",OK,Bestand_2024
2,Zustand A gespeichert und unverändert (Ausgabe...,OK,OK (SHA-256 der Ausgaben geprüft)
3,"Zustand B gültig (Typen, Nichtnegativität, FW/...",OK,Transition_2040_ambitioniert
4,Zustand B gespeichert und unverändert (Ausgabe...,OK,OK (SHA-256 der Ausgaben geprüft)
5,Zeitreihenfolge (Jahr B > Jahr A),OK,2024 → 2040
6,Gesamtbilanz T_A − D + N = T_B,OK,"A 889,872,558.00 m² − Abriss 19,872,558.00 m² ..."
7,Nichtnegativität der Flüsse,OK,min 2.60084e-08 m²
8,Unveränderliche Eigenschaft Geometrie,OK,
9,Unveränderliche Eigenschaft Wetter,OK,


,Block,NGF_A_m2,NGF_B_m2,Kanten_zulaessig,Bestandsfluss_m2,Abriss_m2,Neubau_m2,Kanten_belegt,Status,Max_Herkunftsfehler_m2,Max_Zielfehler_m2
0,EFH_1G | Innsbruck | Leicht,2.258337e+06,2.410929e+06,6660,2.204021e+06,5.431555e+04,2.069082e+05,367,OK,NaN,NaN
1,EFH_1G | Innsbruck | mittel,4.166829e+06,4.448376e+06,6660,4.037356e+06,1.294732e+05,4.110200e+05,369,OK,NaN,NaN
2,EFH_1G | Innsbruck | schwer,4.166829e+06,4.448376e+06,6660,4.139598e+06,2.723157e+04,3.087784e+05,364,OK,NaN,NaN
3,EFH_1G | Wien | Leicht,8.000337e+06,8.540908e+06,6660,7.385356e+06,6.149805e+05,1.155552e+06,358,OK,NaN,NaN
4,EFH_1G | Wien | mittel,1.476414e+07,1.576174e+07,6660,1.474978e+07,1.436346e+04,1.011956e+06,375,OK,NaN,NaN
5,EFH_1G | Wien | schwer,1.476414e+07,1.576174e+07,6660,1.376521e+07,9.989341e+05,1.996527e+06,369,OK,NaN,NaN
6,EFH_2G | Innsbruck | Leicht,9.539268e+06,1.018382e+07,6660,9.528883e+06,1.038485e+04,6.549398e+05,372,OK,NaN,NaN
7,EFH_2G | Innsbruck | mittel,1.747516e+07,1.865593e+07,6660,1.733679e+07,1.383715e+05,1.319144e+06,370,OK,NaN,NaN
8,EFH_2G | Innsbruck | schwer,1.747516e+07,1.865593e+07,6660,1.747433e+07,8.339625e+02,1.181606e+06,372,OK,NaN,NaN
9,EFH_2G | Wien | Leicht,1.711196e+07,1.826819e+07,6660,1.690907e+07,2.028883e+05,1.359120e+06,366,OK,NaN,NaN


## 6 Flüsse, Matrizen und Sankey

- **Was:** Übergänge nach Qualität, Versorgung, PV und Flexibilität.
- **Eingaben:** Flusstabelle
- **Rechnet oder liest:** rechnet Aggregationen
- **Annahmen:** –
- **Interpretation:** Flüsse sind eine zulässige Zuordnung zwischen Zuständen, keine beobachteten Maßnahmen.
- **Sperrende Fehler:** Status ≠ MACHBAR: keine Darstellung

In [9]:
if ergebnis['status'] == fs.MACHBAR:
    for merkmal in ['quality', 'energy', 'pv', 'flex']:
        print(fs.LABEL[merkmal]); display((fs.matrix(ergebnis, merkmal) / 1e6).round(3))
    fs.sankey_auswahl(ergebnis).show()
else:
    print('Keine gültigen Flüsse (Status', ergebnis['status'] + ').')

Qualität


post_quality,vor1919,1919-44,1945-59,1960-79,1980-89,1990-99,Neu_10cm,Neu_20cm,Neu_30cm
pre_quality,,,,,,,,,
vor1919,19.0,0.0,0.0,0.0,0.0,0.000,0.000,13.905,48.907
1919-44,0.0,19.0,0.0,0.0,0.0,0.000,0.000,2.675,8.238
1945-59,0.0,0.0,19.0,0.0,0.0,0.000,0.000,6.959,17.328
1960-79,0.0,0.0,0.0,0.0,0.0,0.000,0.000,26.925,108.366
1980-89,0.0,0.0,0.0,0.0,0.0,0.000,0.000,13.863,60.476
1990-99,0.0,0.0,0.0,0.0,0.0,78.665,0.000,0.000,0.000
Neu_10cm,0.0,0.0,0.0,0.0,0.0,0.000,57.931,0.000,0.000
Neu_20cm,0.0,0.0,0.0,0.0,0.0,0.000,0.000,245.249,0.000
Neu_30cm,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000,123.514


Versorgung


post_energy,WP_Sole,WP_Wasser,WP_Luft,WP_Sole_fc,DH,Bio,Erdgas,Oel,Elektro
pre_energy,,,,,,,,,
WP_Sole,12.744,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0
WP_Wasser,0.000,4.065,0.000,0.000,0.000,0.000,0.0,0.0,0.0
WP_Luft,0.000,0.000,68.333,0.000,0.000,0.000,0.0,0.0,0.0
WP_Sole_fc,0.000,0.000,0.000,12.744,0.000,0.000,0.0,0.0,0.0
DH,2.085,1.905,2.324,1.911,258.621,0.000,0.0,0.0,0.0
Bio,5.489,5.566,5.140,5.034,1.531,137.080,0.0,0.0,0.0
Erdgas,39.130,41.598,44.176,46.503,10.845,3.133,0.0,0.0,0.0
Oel,25.138,23.560,24.820,23.768,8.613,1.726,0.0,0.0,0.0
Elektro,13.252,10.898,12.637,12.738,2.893,0.000,0.0,0.0,0.0


PV


post_pv,PV0,PV30,PV70,PV90
pre_pv,,,,
PV0,432.26,52.038,86.819,124.511
PV30,0.00,82.395,23.764,25.103
PV70,0.00,0.000,19.657,6.557
PV90,0.00,0.000,0.000,16.896


Flexibilität


post_flex,noFlex,Flex
pre_flex,,
noFlex,864.814,4.549
Flex,0.000,0.637


c:\Users\ma0026.NB-MA0026\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




## 7 Änderungsanteile, Neubau und Abriss

- **Was:** Geänderte Bestandsfläche je Typmerkmal (ohne Richtung) und insgesamt; Neubau und Abriss gesondert; Zusammensetzung je Qualität und Versorgung.
- **Eingaben:** Flusstabelle, Neubau- und Abrisstabelle
- **Rechnet oder liest:** rechnet Summen
- **Annahmen:** Nenner Ausgangs-NGF T_A; jede Bestandsfläche zählt je Merkmal einmal; Neubau/Abriss nie als Merkmalsänderung. Zusammensetzung: A = Abriss + verbleibend, B = aus Bestand + Neubau.
- **Interpretation:** Anteile einer zulässigen LP-Zuordnung, keine beobachteten Maßnahmen; die Auswahl der Abriss-/Neubautypen innerhalb der Freigaben ist nicht eindeutig (bei gleicher unveränderter Fläche mit minimalem Versorgungswechsel).
- **Sperrende Fehler:** Status ≠ MACHBAR

In [10]:
if ergebnis['status'] == fs.MACHBAR:
    display(fs.aenderungsanteile(ergebnis))
    for merkmal in ['quality', 'energy']:
        display(fs.zusammensetzung(ergebnis, merkmal))
    if len(ergebnis.get('abriss', [])) or len(ergebnis.get('neubau', [])):
        print('Abriss', round(ergebnis['abriss'].NGF_m2.sum() / 1e6, 3), 'Mio. m² | Neubau', round(ergebnis['neubau'].NGF_m2.sum() / 1e6, 3), 'Mio. m²')
else:
    print('Keine Änderungsanteile ohne gültige Flüsse.')

,Merkmal,Geaenderte_NGF_m2,Anteil_Ausgangs_NGF_pct,Mittel_pct_pro_Jahr
0,Geometrie,0.000000e+00,0.000000,0.000000
1,Qualität,3.076414e+08,34.571397,2.160712
2,Versorgung,3.764134e+08,42.299696,2.643731
3,PV,3.187914e+08,35.824393,2.239025
4,Flexibilität,4.548995e+06,0.511196,0.031950
5,Wetter,0.000000e+00,0.000000,0.000000
6,Speichermasse,0.000000e+00,0.000000,0.000000
7,Beliebige Änderung (ohne Doppelzählung),5.215987e+08,58.614993,3.663437
8,Abriss (explizit),1.987256e+07,2.233191,0.139574
9,Neubau (explizit),8.000000e+07,8.990051,0.561878


,Qualität,A_gesamt_m2,A_verbleibend_m2,Abriss_m2,B_aus_Bestand_m2,Neubau_m2,B_gesamt_m2,Abriss_pct_von_A,Neubau_pct_von_B
0,vor1919,8.702954e+07,8.181233e+07,5.217206e+06,1.900000e+07,0.000000e+00,1.900000e+07,5.994753,0.000000
1,1919-44,3.016668e+07,2.991293e+07,2.537455e+05,1.900000e+07,0.000000e+00,1.900000e+07,0.841145,0.000000
2,1945-59,4.449363e+07,4.328665e+07,1.206975e+06,1.900000e+07,0.000000e+00,1.900000e+07,2.712692,0.000000
3,1960-79,1.440704e+08,1.352906e+08,8.779759e+06,0.000000e+00,0.000000e+00,0.000000e+00,6.094077,NaN
4,1980-89,7.875372e+07,7.433885e+07,4.414873e+06,0.000000e+00,0.000000e+00,0.000000e+00,5.605923,NaN
5,1990-99,7.866473e+07,7.866473e+07,0.000000e+00,7.866473e+07,0.000000e+00,7.866473e+07,0.000000,0.000000
6,Neu_10cm,5.793070e+07,5.793070e+07,0.000000e+00,5.793070e+07,0.000000e+00,5.793070e+07,0.000000,0.000000
7,Neu_20cm,2.452489e+08,2.452489e+08,0.000000e+00,3.095756e+08,2.292440e+07,3.325000e+08,0.000000,6.894558
8,Neu_30cm,1.235143e+08,1.235143e+08,0.000000e+00,3.668290e+08,5.707560e+07,4.239046e+08,0.000000,13.464256


,Versorgung,A_gesamt_m2,A_verbleibend_m2,Abriss_m2,B_aus_Bestand_m2,Neubau_m2,B_gesamt_m2,Abriss_pct_von_A,Neubau_pct_von_B
0,WP_Sole,1.274362e+07,1.274362e+07,0.000000e+00,9.783811e+07,2.105921e+07,1.188973e+08,0.000000,17.712101
1,WP_Wasser,4.065253e+06,4.065253e+06,0.000000e+00,8.759208e+07,2.262702e+07,1.102191e+08,0.000000,20.529129
2,WP_Luft,6.833349e+07,6.833349e+07,0.000000e+00,1.574307e+08,1.705558e+07,1.744862e+08,0.000000,9.774743
3,WP_Sole_fc,1.274362e+07,1.274362e+07,0.000000e+00,1.026974e+08,1.619995e+07,1.188973e+08,0.000000,13.625161
4,DH,2.669618e+08,2.668464e+08,1.153679e+05,2.825031e+08,2.496902e+06,2.850000e+08,0.043215,0.876106
5,Bio,1.601771e+08,1.598402e+08,3.368283e+05,1.419387e+08,5.613276e+05,1.425000e+08,0.210285,0.393914
6,Erdgas,1.957720e+08,1.853849e+08,1.038703e+07,0.000000e+00,0.000000e+00,0.000000e+00,5.305678,NaN
7,Oel,1.156834e+08,1.076246e+08,8.058785e+06,0.000000e+00,0.000000e+00,0.000000e+00,6.966240,NaN
8,Elektro,5.339235e+07,5.241781e+07,9.745470e+05,0.000000e+00,0.000000e+00,0.000000e+00,1.825256,NaN


Abriss 19.873 Mio. m² | Neubau 80.0 Mio. m²


## 8 Maßnahmenraten und Maßnahmenkombinationen

- **Was:** Thermische Sanierung, Versorgungswechsel, Dekarbonisierung, PV-Ausbau, Flexibilisierung im Bestand sowie Neubau- und Abrissrate; überschneidungsfreie Kombinationen; Versorgungswechsel nach Gruppen; Fossilbilanz; Merkmale von Neubau und Abriss.
- **Eingaben:** Flusstabelle, Neubau-/Abrisstabelle, `config/flaechenstroeme/massnahmen.json`
- **Rechnet oder liest:** rechnet Summen aus vorhandenen Flüssen, keine Optimierung
- **Annahmen:** Rate \(r_M = 100 \cdot A_M / (T_A \cdot (J_B - J_A))\) in % der Ausgangs-NGF pro Jahr (lineare mittlere Periodenrate). Thermische Sanierung = alle zulässigen Qualitätswechsel; Versorgungswechselrate = jeder Wechsel der Versorgungskategorie (umgangssprachlich „Kesseltauschrate“, Kesseltausch innerhalb einer Kategorie nicht erkennbar); Dekarbonisierungsrate = Wechsel fossil (Erdgas, Öl) → erneuerbar bzw. nicht fossil (Biomasse, Fernwärme, Wärmepumpe, Strom direkt), Teilmenge der Versorgungswechsel, ohne Erdgas ↔ Öl, ohne Abriss und Neubau; Fossilbilanz: \(F_A = \text{Abriss} + \text{Dekarbonisierung} + \text{innerhalb fossil} + \text{unverändert}\), zusätzlich bezogen auf die fossile Ausgangsfläche \(F_A\); PV-Ausbau = höhere PV-Kategorie (NGF, nicht kWp); Flexibilisierung = noFlex → Flex. Rückentwicklungen getrennt. Neubau getrennt vom Bestand.
- **Interpretation:** Einzelraten überschneiden sich und dürfen nicht addiert werden; maßgeblich für die Gesamtänderung ist „mindestens eine Änderung“ bzw. die Kombinationstabelle. Rekonstruktionsbasierte Raten einer zulässigen Zuordnung (maximal unverändert, dann minimaler Versorgungswechsel), keine beobachteten Bruttomaßnahmenraten. Die Versorgungswechselrate ist damit die kleinste mit der maximal unveränderten Fläche verträgliche. Fernwärme und Strom gelten als erneuerbar; ihr Erzeugungsmix ist nicht abgebildet. Der Rückgang fossiler Fläche entsteht aus Dekarbonisierung und Abriss – beide stehen getrennt in der Fossilbilanz. Gleiche Jahre: keine Jahresrate.
- **Sperrende Fehler:** Status ≠ MACHBAR; ungültige `massnahmen.json`; Kontrollen mit FEHLER

In [11]:
if ergebnis['status'] == fs.MACHBAR:
    massnahmen = ms.lade_definition(PROJEKT)
    display(ms.pruefe_gegen_regeln(massnahmen, regeln))
    raten = ms.massnahmenraten(ergebnis, massnahmen)
    print(raten.attrs['hinweis'])
    print('Nenner T_A:', round(raten.attrs['nenner_m2'] / 1e6, 3), 'Mio. m² | Zeitspanne:', raten.attrs['jahre'], 'Jahre')
    display(raten)
    display(ms.massnahmenkombinationen(ergebnis, massnahmen))
    display((ms.versorgungswechsel_matrix(ergebnis, massnahmen) / 1e6).round(3))
    display(ms.fossilbilanz(ergebnis, massnahmen))
    if ergebnis.get('extern') is not None:
        display(ms.extern_merkmale(ergebnis, massnahmen, 'Neubau'))
        display(ms.extern_merkmale(ergebnis, massnahmen, 'Abriss'))
    display(ms.kontrollen(ergebnis, massnahmen))
else:
    print('Keine Maßnahmenraten ohne gültige Flüsse.')

,Pruefung,Status,Detail
0,Sanierungsklassifikation = zulässige Qualitäts...,HINWEIS,"10 Paare klassifiziert, 15 zulässig; zulässig,..."


Rekonstruktionsbasierte Raten einer ausgewählten zulässigen Zuordnung zwischen zwei Endzuständen (maximal unveränderte Fläche, bei Gleichheit minimaler Versorgungswechsel). Zwischenzeitliche Mehrfachwechsel, wiederholte Sanierungen oder Neubau mit anschließendem Abriss sind nicht identifizierbar; die Werte sind keine beobachteten Bruttomaßnahmenraten. Einzelraten nicht addieren – dieselbe Fläche kann mehrere Maßnahmen zugleich erhalten (siehe Kombinationen und „mindestens eine Änderung“). PV bezeichnet betroffene NGF, nicht kWp.
Nenner T_A: 889.873 Mio. m² | Zeitspanne: 16 Jahre


,Bereich,Ebene,Massnahme,Flaeche_m2,Anteil_Ausgangs_NGF_pct,Rate_pct_pro_Jahr,Flaechenbasis
0,Bestand,Rate,Thermische Sanierungsrate,3.076414e+08,3.457140e+01,2.160712e+00,Bestandsflüsse mit als Verbesserung klassifizi...
1,Bestand,davon,davon Zielqualität Neu_20cm,6.432672e+07,7.228756e+00,4.517973e-01,Sanierungsflüsse nach Zielqualität
2,Bestand,davon,davon Zielqualität Neu_30cm,2.433147e+08,2.734264e+01,1.708915e+00,Sanierungsflüsse nach Zielqualität
3,Bestand,Rückwärts,Qualitätswechsel nicht als Sanierung klassifiz...,1.120105e-05,1.258725e-12,7.867034e-14,"getrennt ausgewiesen, nicht mit Sanierung verr..."
4,Bestand,Rate,Versorgungswechselrate,3.764134e+08,4.229970e+01,2.643731e+00,Bestandsflüsse mit Änderung der Versorgungskat...
5,Bestand,davon,davon nach Wärmepumpe,3.476722e+08,3.906989e+01,2.441868e+00,Versorgungswechsel nach Zielgruppe
6,Bestand,davon,davon nach Fernwärme,2.388226e+07,2.683784e+00,1.677365e-01,Versorgungswechsel nach Zielgruppe
7,Bestand,davon,davon nach Biomasse,4.858882e+06,5.460200e-01,3.412625e-02,Versorgungswechsel nach Zielgruppe
8,Bestand,davon,davon nach Fossil (Erdgas/Öl),0.000000e+00,0.000000e+00,0.000000e+00,Versorgungswechsel nach Zielgruppe
9,Bestand,davon,davon nach Strom direkt,0.000000e+00,0.000000e+00,0.000000e+00,Versorgungswechsel nach Zielgruppe


,Kombination,Anzahl_Massnahmen,Sanierung,Versorgungswechsel,PV_Ausbau,Flexibilisierung,Sonstige_Aenderung,Flaeche_m2,Anteil_Ausgangs_NGF_pct,Rate_pct_pro_Jahr
0,keine Maßnahme,0,False,False,False,False,False,3.484013e+08,3.915182e+01,NaN
1,sonstige Änderung,0,False,False,False,False,True,4.711009e-08,5.294027e-15,3.308767e-16
2,Versorgungswechsel,1,False,True,False,False,False,7.024128e+07,7.893409e+00,4.933380e-01
3,PV-Ausbau,1,False,False,True,False,False,5.526568e+07,6.210517e+00,3.881573e-01
4,Sanierung,1,True,False,False,False,False,2.666830e+07,2.996868e+00,1.873042e-01
5,Versorgungswechsel + sonstige Änderung,1,False,True,False,False,True,6.345753e-06,7.131081e-13,4.456926e-14
6,PV-Ausbau + sonstige Änderung,1,False,False,True,False,True,6.075851e-07,6.827776e-14,4.267360e-15
7,Sanierung + Versorgungswechsel,2,True,True,False,False,False,1.030728e+08,1.158287e+01,7.239295e-01
8,Versorgungswechsel + PV-Ausbau,2,False,True,True,False,False,8.642834e+07,9.712441e+00,6.070275e-01
9,Sanierung + PV-Ausbau,2,True,False,True,False,False,6.325137e+07,7.107913e+00,4.442446e-01


Ziel,Wärmepumpe,Fernwärme,Biomasse,Fossil (Erdgas/Öl),Strom direkt
Herkunft,,,,,
Wärmepumpe,0.000,0.000,0.000,0.0,0.0
Fernwärme,8.226,0.000,0.000,0.0,0.0
Biomasse,21.229,1.531,0.000,0.0,0.0
Fossil (Erdgas/Öl),268.692,19.458,4.859,0.0,0.0
Strom direkt,49.525,2.893,0.000,0.0,0.0


,Bereich,Teil,Position,Flaeche_m2,Anteil_Ausgangs_NGF_pct,Rate_pct_pro_Jahr,Anteil_fossile_Ausgangsflaeche_pct,Rate_fossil_pct_pro_Jahr
0,Fossilbilanz,Ausgangszustand,Fossil versorgte Fläche im Ausgangszustand F_A,3.114554e+08,35.00000,NaN,100.000000,NaN
1,Fossilbilanz,Verbleib F_A,davon abgerissen,1.844581e+07,2.07286,0.129554,5.922458,0.370154
2,Fossilbilanz,Verbleib F_A,davon Wechsel zu erneuerbar (Dekarbonisierung),2.930096e+08,32.92714,2.057946,94.077542,5.879846
3,Fossilbilanz,Verbleib F_A,davon Wechsel innerhalb fossil (Erdgas ↔ Öl),0.000000e+00,0.00000,NaN,0.000000,NaN
4,Fossilbilanz,Verbleib F_A,davon fossil unverändert,0.000000e+00,0.00000,NaN,0.000000,NaN
5,Fossilbilanz,Herkunft F_B,Wechsel erneuerbar → fossil,0.000000e+00,0.00000,0.000000,0.000000,0.000000
6,Fossilbilanz,Herkunft F_B,Neubau fossil,0.000000e+00,0.00000,0.000000,0.000000,0.000000
7,Fossilbilanz,Zielzustand,Fossil versorgte Fläche im Zielzustand F_B,0.000000e+00,0.00000,NaN,0.000000,NaN
8,Fossilbilanz,Saldo,Rückgang fossil versorgter Fläche F_A − F_B,3.114554e+08,35.00000,2.187500,100.000000,6.250000


,Art,Merkmal,Kategorie,Flaeche_m2,Anteil_Neubau_pct,Rate_pct_pro_Jahr
0,Neubau,Qualität,Neu_20cm,2.292440e+07,28.655505,0.161009
1,Neubau,Qualität,Neu_30cm,5.707560e+07,71.344495,0.400869
2,Neubau,Versorgungsgruppe,Wärmepumpe,7.694177e+07,96.177213,0.540399
3,Neubau,Versorgungsgruppe,Fernwärme,2.496902e+06,3.121127,0.017537
4,Neubau,Versorgungsgruppe,Biomasse,5.613276e+05,0.701660,0.003942
5,Neubau,PV,mit PV (> PV0),3.726019e+07,46.575236,0.261696
6,Neubau,Flexibilität,Flex,7.586904e+05,0.948363,0.005329
7,Neubau,Gesamt,Neubau,8.000000e+07,100.000000,0.561878


,Art,Merkmal,Kategorie,Flaeche_m2,Anteil_Abriss_pct,Rate_pct_pro_Jahr
0,Abriss,Qualität,vor1919,5.217206e+06,26.253317,0.036643
1,Abriss,Qualität,1919-44,2.537455e+05,1.276864,0.001782
2,Abriss,Qualität,1945-59,1.206975e+06,6.073576,0.008477
3,Abriss,Qualität,1960-79,8.779759e+06,44.180315,0.061664
4,Abriss,Qualität,1980-89,4.414873e+06,22.215928,0.031008
5,Abriss,Versorgungsgruppe,Fernwärme,1.153679e+05,0.580539,0.000810
6,Abriss,Versorgungsgruppe,Biomasse,3.368283e+05,1.694942,0.002366
7,Abriss,Versorgungsgruppe,Fossil (Erdgas/Öl),1.844581e+07,92.820536,0.129554
8,Abriss,Versorgungsgruppe,Strom direkt,9.745470e+05,4.903984,0.006845
9,Abriss,PV,mit PV (> PV0),3.602128e+06,18.126139,0.025299


,Pruefung,Status,Detail
0,Nenner T_A = Bestandsflüsse + Abriss,OK,"T_A = 889,872,558.00 m²"
1,Nenner T_A = NGF des Ausgangszustands,OK,"889,872,558.00 vs. 889,872,558.00 m²"
2,Kombinationen decken die Bestandsfläche genau ...,OK,"870,000,000.00 vs. 870,000,000.00 m²"
3,Mindestens eine Änderung + ohne Änderung = Bes...,OK,"521,598,741.72 + 348,401,258.28 m²"
4,Jede Einzelmaßnahme ≤ mindestens eine Änderung,OK,"Sanierung 307,641,374; Versorgungswechsel 376,..."
5,Dekarbonisierung ≤ Versorgungswechsel,OK,"293,009,581 ≤ 376,413,387 m²"
6,Fossilbilanz Verbleib F_A geschlossen,OK,"311,455,395.30 vs. 311,455,395.30 m²"
7,Fossilbilanz Herkunft F_B geschlossen,OK,0.00 vs. 0.00 m²
8,Unveränderliche Merkmale ohne Wechsel,OK,"Geometrie, Wetter, Speichermasse"
9,Zeitspanne für Jahresraten,OK,16 Jahre


## 9 Wiederholbarkeit (Abnahme)

- **Was:** Vergleicht eine in dieser Sitzung neu gerechnete Rekonstruktion mit dem gespeicherten Ergebnis desselben Paares – vor einem Ersetzen.
- **Eingaben:** neues Ergebnis dieser Sitzung, `results/flaechenstroeme/<REKONSTRUKTION>/`
- **Rechnet oder liest:** liest und vergleicht, keine Optimierung; nur wenn in Abschnitt 4–5 neu rekonstruiert wurde
- **Annahmen:** Toleranz max(1 m², 1e-7 · T_A) entsprechend der relativen HiGHS-Toleranz; maßgeblich sind Zielwerte, Neubau/Abriss und Raten.
- **Interpretation:** OK: innerhalb der Toleranz wiederholbar. HINWEIS bei „Flüsse je Typpaar“: gleichwertige alternative Zuordnung (nicht eindeutig), Zielwerte entscheiden. FEHLER: Status, Zielwerte oder Neubau/Abriss weichen ab.
- **Sperrende Fehler:** –

In [12]:
ordner = PROJEKT['ergebnisse']['flaechenstroeme'] / REKONSTRUKTION
if ergebnis.get('quelle') == 'rekonstruiert' and ordner.is_dir():
    display(ab.wiederholbarkeit(ergebnis, fs.lade_flussergebnis(REKONSTRUKTION, PROJEKT), ms.lade_definition(PROJEKT)))
else:
    print('Kein Vergleich: nur nach neuer Rekonstruktion und vorhandenem gespeichertem Ergebnis.')

Kein Vergleich: nur nach neuer Rekonstruktion und vorhandenem gespeichertem Ergebnis.


## 10 Speichern und wieder laden

- **Was:** Speichert das Ergebnis geprüft (auch NICHT_MACHBAR mit Diagnose; bei MACHBAR einschließlich Maßnahmenraten) und liest es zur Kontrolle wieder ein.
- **Eingaben:** Ergebnis
- **Rechnet oder liest:** schreibt nur bei `EXPORTIEREN=True`
- **Annahmen:** Arbeitsordner, dann Veröffentlichung; vorhandener Ordner nur mit `ERSETZEN=True` ersetzt.
- **Interpretation:** –
- **Sperrende Fehler:** Ordner existiert und `ERSETZEN=False`; Status NICHT_AUSGEFUEHRT

In [13]:
if EXPORTIEREN:
    ausgabe = fs.speichere_ergebnis(ergebnis, PROJEKT, ersetzen=ERSETZEN)
    geladen = fs.lade_flussergebnis(REKONSTRUKTION, PROJEKT)
    print('Gespeichert:', ausgabe, '| wieder geladen, Status', geladen['status'])
else:
    print('EXPORTIEREN=False: nichts gespeichert.')

Gespeichert: results/flaechenstroeme/Bestand_2024__Transition_2040_ambitioniert | wieder geladen, Status MACHBAR
